# NIFTY Expiry Day Volatility Research (Focus: 2023 – 2025)

## 1. Research Motivation & Scope
In recent years, the Indian derivatives market (NSE) experienced exponential growth in Index Options trading, driven by the proliferation of weekly expiries, retail participation, and systematic zero-days-to-expiration (0DTE) strategies. 

A central question in financial market microstructure and quantitative finance is:
> **Did NIFTY option expiry days exhibit abnormal volatility between 2023 and 2025 compared to normal trading days?**

Two competing market microstructure hypotheses exist:
1. **The Volatility Amplification Hypothesis (Delta/Gamma Hedging & Expiry Chaos)**: As options near expiration, gamma spikes toward infinity for at-the-money contracts. Market makers dynamically rehedging their short gamma positions must buy as prices rise and sell as prices fall, exacerbating market swings and inflating intraday volatility.
2. **The Volatility Dampening / Gamma Pinning Hypothesis**: Heavy concentration of open interest at key strikes can create an attractive "pinning" force as expiry approaches, while aggressive index option sellers (straddle/strangle decay harvesters) suppress intraday realized volatility.

In this notebook, we systematically evaluate these hypotheses using clean historical daily NIFTY market data merged with official exchange expiry records.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import warnings
warnings.filterwarnings('ignore')

# Set styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12


## 2. Data Ingestion & Overview
We load the curated dataset `nifty_data_with_expiries.csv`, which contains:
* Daily NIFTY OHLC prices and volumes (2016 – 2026)
* Ground-truth `is_expiry` flags matching verified exchange records
* `expiry_type` classification (`Monthly`, `Weekly`, `Non-Expiry`)
* Calendar features (`day_name`, `year`, `month`)


In [2]:
# Load the freshly curated dataset (handles running from notebooks/ or project root)
data_path = 'data/processed/nifty_data_with_expiries.csv' if os.path.exists('data/processed/nifty_data_with_expiries.csv') else '../data/processed/nifty_data_with_expiries.csv'
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"Total Trading Days Loaded: {len(df)}")
print(f"Date Span: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Total Expiry Days: {df['is_expiry'].sum()} (Monthly: {(df['expiry_type']=='Monthly').sum()}, Weekly: {(df['expiry_type']=='Weekly').sum()})")
df.head(5)


Total Trading Days Loaded: 2599
Date Span: 2016-01-01 to 2026-07-24
Total Expiry Days: 429 (Monthly: 127, Weekly: 302)


,Date,Symbol,Futures Contracts,Futures Quantity,Futures Value (Rs. In Crs.),Options Contracts,Options Quantity,Options Value (Rs. In Crs.),Total Overall Value (Rs. In Crs.),Equity Traded Value (Rs. In Crs.),...,open,high,low,close,is_expiry,day_name,day_of_week,year,month,expiry_type
0,2016-01-01,NIFTY,81885,6141375,4890.54,859166,64437450,52040.85,56931.39,12790.77,...,7938.45,7972.55,7909.80,7963.20,False,Friday,4,2016,1,Non-Expiry
1,2016-01-04,NIFTY,226345,16975875,13340.09,2546999,191024925,151788.40,165128.49,17179.35,...,7924.55,7937.55,7781.10,7791.30,False,Monday,0,2016,1,Non-Expiry
2,2016-01-05,NIFTY,148669,11150175,8713.19,1945485,145911375,115647.05,124360.24,17657.69,...,7828.40,7831.20,7763.25,7784.65,False,Tuesday,1,2016,1,Non-Expiry
3,2016-01-06,NIFTY,169347,12701025,9877.81,2131230,159842250,126551.65,136429.46,20110.22,...,7788.05,7800.95,7721.20,7741.00,False,Wednesday,2,2016,1,Non-Expiry
4,2016-01-07,NIFTY,234900,17617500,13419.15,3324174,249313050,193468.48,206887.63,18650.32,...,7673.35,7674.95,7556.60,7568.30,False,Thursday,3,2016,1,Non-Expiry


## 3. Mathematical Formulation of the 4 Volatility Estimators

We compute the 4 volatility proxies for each trading day:

1. **Intraday Range**:
   \\text{Intraday Range}_t = \\frac{\\text{High}_t - \\text{Low}_t}{\\text{Open}_t}

2. **Absolute Daily Return**:
   \\text{Abs Return}_t = \\frac{|\\text{Close}_t - \\text{Close}_{t-1}|}{\\text{Close}_{t-1}}

3. **Parkinson Variance (Daily)**:
   \\sigma_{P, t}^2 = \\frac{1}{4 \\ln(2)} \\left(\\ln\\frac{\\text{High}_t}{\\text{Low}_t}\\right)^2

4. **Garman-Klass Variance (Daily)**:
   \\sigma_{GK, t}^2 = 0.5 \\left(\\ln\\frac{\\text{High}_t}{\\text{Low}_t}\\right)^2 - (2\\ln(2) - 1) \\left(\\ln\\frac{\\text{Close}_t}{\\text{Open}_t}\\right)^2

*Note: For Parkinson and Garman-Klass, we work in variance space to prevent Jensen's inequality bias, and compute annualized standard deviations ($\\sqrt{\\sigma^2 \\times 252}$) for intuitive reporting.*

In [3]:
# 1. Compute Previous Close
df['prev_close'] = df['close'].shift(1)
df = df.dropna(subset=['prev_close']).copy().reset_index(drop=True)

# 2. Metric 1: Intraday Range
df['intraday_range'] = (df['high'] - df['low']) / df['open']

# 3. Metric 2: Absolute Daily Return
df['abs_daily_return'] = np.abs(df['close'] - df['prev_close']) / df['prev_close']

# Logarithmic price ratios
ln_hl = np.log(df['high'] / df['low'])
ln_co = np.log(df['close'] / df['open'])

# 4. Metric 3: Parkinson Variance (Daily)
df['parkinson_var'] = (1 / (4 * np.log(2))) * (ln_hl ** 2)

# 5. Metric 4: Garman-Klass Variance (Daily)
df['garman_klass_var'] = 0.5 * (ln_hl ** 2) - (2 * np.log(2) - 1) * (ln_co ** 2)

# Annualized Volatilities (assuming 252 trading days)
df['parkinson_vol_annualized'] = np.sqrt(df['parkinson_var'] * 252)
df['garman_klass_vol_annualized'] = np.sqrt(df['garman_klass_var'] * 252)

vol_cols = [
    'intraday_range', 
    'abs_daily_return', 
    'parkinson_var', 
    'garman_klass_var', 
    'parkinson_vol_annualized', 
    'garman_klass_vol_annualized'
]

display(df[['Date', 'is_expiry', 'expiry_type'] + vol_cols].head())


,Date,is_expiry,expiry_type,intraday_range,abs_daily_return,parkinson_var,garman_klass_var,parkinson_vol_annualized,garman_klass_vol_annualized
0,2016-01-04,False,Non-Expiry,0.019742,0.021587,0.000143,0.000087,0.189785,0.148117
1,2016-01-05,False,Non-Expiry,0.008680,0.000854,0.000027,0.000026,0.083083,0.080695
2,2016-01-06,False,Non-Expiry,0.010240,0.005607,0.000038,0.000039,0.097965,0.098641
3,2016-01-07,False,Non-Expiry,0.015424,0.022310,0.000087,0.000047,0.148156,0.109233
4,2016-01-08,False,Non-Expiry,0.006970,0.004367,0.000018,0.000024,0.066481,0.077127


## 4. Weekly Relative Volatility Calculation

To evaluate whether an expiry day concentrates abnormal volatility compared to the surrounding trading week, we compute the **Weekly Relative Volatility**:

1. **Rest-of-Week Volatility (Excluding Expiry)**:
   For each week, we calculate the total volatility of the non-expiry trading days by taking the root of the squared sum of daily volatilities (equivalent to root sum of daily variances):
   $$\sigma_{\text{rest\_of\_week}} = \sqrt{\sum_{i \in \text{non-expiry days}} \sigma_i^2}$$

2. **Weekly Relative Volatility**:
   $$\text{Relative Volatility} = \frac{\sigma_{\text{expiry\_day}}}{\sigma_{\text{rest\_of\_week}}}$$

> **Interpretation & Benchmark**:
> In a standard 5-day trading week (1 expiry day and 4 non-expiry days), if all days experienced identical daily volatility $\sigma$:
> $$\sigma_{\text{rest\_of\_week}} = \sqrt{4 \times \sigma^2} = 2\sigma \implies \text{Relative Volatility} = \frac{\sigma}{2\sigma} = 0.50$$
> * $\text{Relative Volatility} > 0.50$: The expiry day experienced disproportionately elevated volatility relative to the rest of the week.
> * $\text{Relative Volatility} < 0.50$: The expiry day experienced compressed/pinned volatility relative to the rest of the week.

In [4]:
# 1. Create Weekly Identifier (Monday - Sunday)
df['YearWeek'] = df['Date'].dt.to_period('W')

weekly_records = []

for yw, group in df.groupby('YearWeek'):
    exp_days = group[group['is_expiry']]
    non_exp_days = group[~group['is_expiry']]
    
    # Only evaluate weeks with at least one expiry and at least one non-expiry day
    if len(exp_days) == 0 or len(non_exp_days) == 0:
        continue
        
    start_date = group['Date'].min()
    end_date = group['Date'].max()
    year = start_date.year
    
    # -------------------------------------------------------------
    # Root of squared sums for rest of week: sqrt(sum(vol_i^2))
    # Note: For Parkinson & Garman-Klass, vol^2 is the daily variance!
    # -------------------------------------------------------------
    
    # 1. Garman-Klass Volatility
    gk_exp = np.sqrt(exp_days['garman_klass_var'].sum())
    gk_rest = np.sqrt(non_exp_days['garman_klass_var'].sum())
    rel_vol_gk = gk_exp / gk_rest if gk_rest > 0 else np.nan
    
    # 2. Parkinson Volatility
    park_exp = np.sqrt(exp_days['parkinson_var'].sum())
    park_rest = np.sqrt(non_exp_days['parkinson_var'].sum())
    rel_vol_park = park_exp / park_rest if park_rest > 0 else np.nan
    
    # 3. Intraday Normalized Range
    range_exp = np.sqrt((exp_days['intraday_range']**2).sum())
    range_rest = np.sqrt((non_exp_days['intraday_range']**2).sum())
    rel_vol_range = range_exp / range_rest if range_rest > 0 else np.nan
    
    # 4. Absolute Daily Return
    ret_exp = np.sqrt((exp_days['abs_daily_return']**2).sum())
    ret_rest = np.sqrt((non_exp_days['abs_daily_return']**2).sum())
    rel_vol_ret = ret_exp / ret_rest if ret_rest > 0 else np.nan
    
    # Primary expiry day details
    primary_exp = exp_days.iloc[0]
    
    weekly_records.append({
        'YearWeek': str(yw),
        'Week_Start': start_date,
        'Year': year,
        'Expiry_Date': primary_exp['Date'],
        'Expiry_Day': primary_exp['day_name'],
        'Expiry_Type': primary_exp['expiry_type'],
        'Non_Expiry_Count': len(non_exp_days),
        'GK_Expiry_Vol': gk_exp,
        'GK_Rest_Week_Vol': gk_rest,
        'Relative_Vol_GK': rel_vol_gk,
        'Relative_Vol_Parkinson': rel_vol_park,
        'Relative_Vol_Range': rel_vol_range,
        'Relative_Vol_Return': rel_vol_ret
    })

weekly_df = pd.DataFrame(weekly_records)

print(f"Total Weeks Analyzed: {len(weekly_df)}")
print(f"Weeks in 2023-2025 focus period: {len(weekly_df[(weekly_df['Year'] >= 2023) & (weekly_df['Year'] <= 2025)])}")

# Display sample of weekly relative volatility
display(weekly_df[['YearWeek', 'Expiry_Date', 'Expiry_Day', 'Expiry_Type', 'Non_Expiry_Count', 'Relative_Vol_GK', 'Relative_Vol_Parkinson', 'Relative_Vol_Range', 'Relative_Vol_Return']].head(10))


Total Weeks Analyzed: 426
Weeks in 2023-2025 focus period: 157


,YearWeek,Expiry_Date,Expiry_Day,Expiry_Type,Non_Expiry_Count,Relative_Vol_GK,Relative_Vol_Parkinson,Relative_Vol_Range,Relative_Vol_Return
0,2016-01-25/2016-01-31,2016-01-28,Thursday,Monthly,3,0.436735,0.307229,0.305429,0.093685
1,2016-02-22/2016-02-28,2016-02-25,Thursday,Monthly,4,0.361984,0.387528,0.388541,0.293629
2,2016-03-28/2016-04-03,2016-03-31,Thursday,Monthly,4,0.439852,0.348174,0.350200,0.018142
3,2016-04-25/2016-05-01,2016-04-28,Thursday,Monthly,4,0.715199,0.759040,0.750628,1.111121
4,2016-05-23/2016-05-29,2016-05-26,Thursday,Monthly,4,0.835764,0.690241,0.691507,0.640119
5,2016-06-27/2016-07-03,2016-06-30,Thursday,Monthly,4,0.567716,0.522852,0.522018,0.898396
6,2016-07-25/2016-07-31,2016-07-28,Thursday,Monthly,4,0.317758,0.283198,0.282582,0.451810
7,2016-08-22/2016-08-28,2016-08-25,Thursday,Monthly,4,0.622867,0.779956,0.778614,1.256673
8,2016-09-26/2016-10-02,2016-09-29,Thursday,Monthly,4,1.263020,1.513752,1.496253,1.312609
9,2016-10-24/2016-10-30,2016-10-27,Thursday,Monthly,4,0.657551,0.610567,0.610329,0.000000


## 5. Visualizing Weekly Relative Volatility Trend (Altair)

We plot the **Weekly Relative Volatility** over time to visually inspect structural trends and regimes:

* **Points**: Each weekly expiry day (color-coded by `Expiry_Type`: Monthly vs Weekly).
* **Orange Line**: 12-week moving average of Relative Volatility (smoothing short-term noise to reveal underlying trend).
* **Dashed Black Line at $y = 0.50$**: The theoretical neutral parity threshold. Values above $0.50$ indicate that the expiry day realized more volatility than an average non-expiry day in that week.
* **Interactive**: Use click & drag to pan horizontally across time, or scroll to zoom in/out (e.g. to examine 2023–2025 closely). Hover over any point to view exact dates, ratios, and expiry details.

In [5]:
import altair as alt

# 1. Prepare plotting dataframe
plot_df = weekly_df.copy()
# Ensure Date is timestamp for Altair time-scale
plot_df['Date'] = pd.to_datetime(plot_df['Expiry_Date'])

# Calculate 12-week rolling moving average of relative volatility
plot_df['Relative_Vol_GK_12w_MA'] = plot_df['Relative_Vol_GK'].rolling(12, min_periods=3).mean()

# 2. Base scatter points for each expiry day
points = alt.Chart(plot_df).mark_circle(size=45, opacity=0.7).encode(
    x=alt.X('Date:T', title='Expiry Date'),
    y=alt.Y('Relative_Vol_GK:Q', title='Relative Volatility (Expiry Vol / Rest of Week Vol)'),
    color=alt.Color(
        'Expiry_Type:N',
        scale=alt.Scale(domain=['Weekly', 'Monthly'], range=['#1f77b4', '#d62728']),
        title='Expiry Type'
    ),
    tooltip=[
        alt.Tooltip('Date:T', title='Expiry Date', format='%Y-%m-%d'),
        alt.Tooltip('Expiry_Day:N', title='Day of Week'),
        alt.Tooltip('Expiry_Type:N', title='Expiry Type'),
        alt.Tooltip('Relative_Vol_GK:Q', title='Relative Vol (Garman-Klass)', format='.3f'),
        alt.Tooltip('Relative_Vol_Range:Q', title='Relative Vol (Range)', format='.3f'),
        alt.Tooltip('Relative_Vol_GK_12w_MA:Q', title='12-Week MA', format='.3f'),
        alt.Tooltip('Non_Expiry_Count:Q', title='Non-Expiry Days in Week')
    ]
)

# 3. Connecting line between weekly points
line = alt.Chart(plot_df).mark_line(color='#a6cee3', strokeWidth=1, opacity=0.6).encode(
    x='Date:T',
    y='Relative_Vol_GK:Q'
)

# 4. 12-Week Moving Average Trend Line
trend_line = alt.Chart(plot_df).mark_line(color='#ff7f0e', strokeWidth=2.5).encode(
    x='Date:T',
    y=alt.Y('Relative_Vol_GK_12w_MA:Q', title='')
)

# 5. Theoretical Neutral Parity Line (y = 0.50)
parity_rule = alt.Chart(pd.DataFrame({'y': [0.50]})).mark_rule(
    color='black',
    strokeDash=[6, 4],
    strokeWidth=1.5
).encode(
    y='y:Q'
)

# Parity annotation text
parity_text = alt.Chart(pd.DataFrame({'y': [0.50], 'text': ['Neutral Baseline (0.50)']})).mark_text(
    align='left',
    dx=10,
    dy=-8,
    color='#333333',
    fontSize=11,
    fontWeight='bold'
).encode(
    y='y:Q',
    text='text:N'
)

# Combine layers and set interactive properties
chart = (line + points + trend_line + parity_rule + parity_text).properties(
    title='Weekly Relative Volatility Dynamics (2016–2026): Expiry Day vs Rest of Week',
    width=1000,
    height=480
).interactive()

chart


alt.LayerChart(...)

## 6. Policy Impact Evaluation: 2-Regime Segmentation (Test 3: Tail Risk & Outliers)

To evaluate whether SEBI's regulatory intervention on **November 20, 2024** (*discontinuing multi-index weeklies, raising lot sizes, and enforcing upfront margins*) altered market dynamics, we segment the recent data into two balanced regimes:

* **Regime 1 — "Peak 0DTE Frenzy"**: `2023-09-04` to `2024-11-19`
  * Multi-index weekly expiries active Monday through Friday across NSE/BSE.
  * Smallest lot size ($25$), peak speculative retail activity and algorithmic gamma trading.
* **Regime 2 — "Post-SEBI Mandate"**: `2024-11-20` to `2025-12-31`
  * Single weekly expiry on NSE (NIFTY 50 only; Bank Nifty/FinNifty/Midcap weeklies discontinued).
  * Recalibrated larger contract lot sizes ($75 \to 65$) and subsequent migration of weekly expiry to Tuesday.

### Test 3 Objective: Tail Risk & Gamma Spikes
Rather than focusing solely on average volatility (which may remain stable due to macro factors), we test whether the mandate succeeded at its core goal: **eliminating extreme volatility spikes and tail blowouts on expiry days**.

In [6]:
# 1. Assign Regulatory Regimes to the Weekly Data
conditions_regime = [
    (weekly_df['Expiry_Date'] >= '2023-09-04') & (weekly_df['Expiry_Date'] <= '2024-11-19'),
    (weekly_df['Expiry_Date'] >= '2024-11-20') & (weekly_df['Expiry_Date'] <= '2025-12-31')
]
choices_regime = ['Regime 1: Frenzy Era (Pre-Mandate)', 'Regime 2: Post-Mandate']

weekly_df['Regime'] = np.select(conditions_regime, choices_regime, default='Historical / Other')

# Filter to the two comparison regimes
regime_data = weekly_df[weekly_df['Regime'].isin(choices_regime)].copy()

# 2. Function to compute comprehensive tail risk and dispersion metrics
def evaluate_tail_risk(df_group, metric_col):
    records = []
    for regime_name, group in df_group.groupby('Regime'):
        vals = group[metric_col].dropna()
        records.append({
            'Regime': regime_name,
            'Weeks (N)': len(vals),
            'Mean': vals.mean(),
            'Std Dev': vals.std(),
            'Median (P50)': vals.median(),
            'IQR (P75-P25)': vals.quantile(0.75) - vals.quantile(0.25),
            '75th Pct': vals.quantile(0.75),
            '90th Pct': vals.quantile(0.90),
            '95th Pct': vals.quantile(0.95),
            'Max Spike': vals.max(),
            'Skewness': stats.skew(vals),
            'Excess Kurtosis': stats.kurtosis(vals),
            'Weeks > 0.75 Ratio (%)': (vals > 0.75).mean() * 100,
            'Weeks > 1.00 Ratio (%)': (vals > 1.00).mean() * 100
        })
    return pd.DataFrame(records)

# 3. Tail Risk Comparison for Garman-Klass Relative Volatility
tail_gk = evaluate_tail_risk(regime_data, 'Relative_Vol_GK')
tail_range = evaluate_tail_risk(regime_data, 'Relative_Vol_Range')

print("=== TEST 3: TAIL RISK & EXTREME VOLATILITY METRICS (Garman-Klass) ===")
display(tail_gk.style.format({
    'Mean': '{:.4f}',
    'Std Dev': '{:.4f}',
    'Median (P50)': '{:.4f}',
    'IQR (P75-P25)': '{:.4f}',
    '75th Pct': '{:.4f}',
    '90th Pct': '{:.4f}',
    '95th Pct': '{:.4f}',
    'Max Spike': '{:.4f}',
    'Skewness': '{:.3f}',
    'Excess Kurtosis': '{:.3f}',
    'Weeks > 0.75 Ratio (%)': '{:.2f}%',
    'Weeks > 1.00 Ratio (%)': '{:.2f}%'
}))

# 4. Formal Statistical Tests between the Two Regimes
r1_gk = regime_data[regime_data['Regime'] == 'Regime 1: Frenzy Era (Pre-Mandate)']['Relative_Vol_GK']
r2_gk = regime_data[regime_data['Regime'] == 'Regime 2: Post-Mandate']['Relative_Vol_GK']

# Tests:
mw_stat, mw_p = stats.mannwhitneyu(r1_gk, r2_gk, alternative='two-sided')
ks_stat, ks_p = stats.ks_2samp(r1_gk, r2_gk)
levene_stat, levene_p = stats.levene(r1_gk, r2_gk)

test_summary = pd.DataFrame([
    {
        'Statistical Test': 'Mann-Whitney U (Central Tendency / Ranks)',
        'Test Statistic': mw_stat,
        'p-value': mw_p,
        'Null Hypothesis (H0)': 'Identical distribution locations',
        'Reject H0 at 5%': 'YES' if mw_p < 0.05 else 'NO'
    },
    {
        'Statistical Test': 'Kolmogorov-Smirnov (Distribution Shape)',
        'Test Statistic': ks_stat,
        'p-value': ks_p,
        'Null Hypothesis (H0)': 'Identical continuous distributions',
        'Reject H0 at 5%': 'YES' if ks_p < 0.05 else 'NO'
    },
    {
        'Statistical Test': 'Levene Test (Variance / Dispersion Equality)',
        'Test Statistic': levene_stat,
        'p-value': levene_p,
        'Null Hypothesis (H0)': 'Equal variances across regimes',
        'Reject H0 at 5%': 'YES' if levene_p < 0.05 else 'NO'
    }
])

print("\n=== HYPOTHESIS TESTING SUMMARY ===")
display(test_summary.style.format({'Test Statistic': '{:.4f}', 'p-value': '{:.4f}'}))


=== TEST 3: TAIL RISK & EXTREME VOLATILITY METRICS (Garman-Klass) ===


,Regime,Weeks (N),Mean,Std Dev,Median (P50),IQR (P75-P25),75th Pct,90th Pct,95th Pct,Max Spike,Skewness,Excess Kurtosis,Weeks > 0.75 Ratio (%),Weeks > 1.00 Ratio (%)
0,Regime 1: Frenzy Era (Pre-Mandate),63,0.5681,0.3171,0.4761,0.3604,0.7084,0.9992,1.1880,1.8551,1.568,3.055,22.22%,11.11%
1,Regime 2: Post-Mandate,59,0.5523,0.2983,0.4724,0.2792,0.6534,0.9016,1.0496,1.8380,1.773,4.633,22.03%,6.78%



=== HYPOTHESIS TESTING SUMMARY ===


,Statistical Test,Test Statistic,p-value,Null Hypothesis (H0),Reject H0 at 5%
0,Mann-Whitney U (Central Tendency / Ranks),1868.0000,0.9632,Identical distribution locations,NO
1,Kolmogorov-Smirnov (Distribution Shape),0.1076,0.8195,Identical continuous distributions,NO
2,Levene Test (Variance / Dispersion Equality),0.2759,0.6004,Equal variances across regimes,NO
